# Iris ML Demo: MLflow, AutoML, and Hyperopt

This notebook demonstrates a complete machine learning workflow using:
- **MLflow** for experiment tracking and model management
- **Databricks AutoML** for automated model selection
- **Hyperopt** for hyperparameter tuning

**Dataset**: Unity Catalog table `main.tomes_gen.iris`  
**Target**: `Species` (multi-class classification)  
**Date**: November 2025


## Phase 1: Setup and Data Preparation


### Step 1.1: Import Required Libraries


In [1]:
# System imports
import sys
import os
sys.path.append(os.path.abspath(".."))

# Spark imports
from spark_env import spark
from pyspark.sql.functions import col, lit
import pyspark.sql.functions as F

# MLflow
import mlflow
from mlflow import MlflowClient

# Databricks AutoML
# import databricks.automl

# Scikit-learn (for Hyperopt tuning)
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# LightGBM (for Hyperopt tuning)
# import lightgbm as lgb
from lightgbm import LGBMClassifier

# Hyperopt
from hyperopt import fmin, tpe, hp, SparkTrials, STATUS_OK, Trials
from hyperopt.pyll import scope

# NumPy and Pandas
import numpy as np
import pandas as pd

print("All libraries imported successfully!")


Driver is running on local environment
All libraries imported successfully!


/opt/homebrew/anaconda3/envs/py312dbx/lib/python3.12/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### Step 1.2: Load Data from Unity Catalog


In [2]:
# Load the dataset from Unity Catalog
df = spark.table("main.tomes_gen.iris")

# Display basic information
print("Dataset shape (rows, columns):")
print(f"Rows: {df.count()}, Columns: {len(df.columns)}")

print("\nFirst few rows:")
df.show(10)

print("\nSchema:")
df.printSchema()

print("\nSummary statistics:")
df.describe().show()

# Verify target column and class distribution
print("\nClass distribution (Species):")
df.groupBy("Species").count().orderBy("Species").show()


Dataset shape (rows, columns):
Rows: 150, Columns: 6

First few rows:
+---+-------------+------------+-------------+------------+-----------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species|
+---+-------------+------------+-------------+------------+-----------+
|  1|          5.1|         3.5|          1.4|         0.2|Iris-setosa|
|  2|          4.9|         3.0|          1.4|         0.2|Iris-setosa|
|  3|          4.7|         3.2|          1.3|         0.2|Iris-setosa|
|  4|          4.6|         3.1|          1.5|         0.2|Iris-setosa|
|  5|          5.0|         3.6|          1.4|         0.2|Iris-setosa|
|  6|          5.4|         3.9|          1.7|         0.4|Iris-setosa|
|  7|          4.6|         3.4|          1.4|         0.3|Iris-setosa|
|  8|          5.0|         3.4|          1.5|         0.2|Iris-setosa|
|  9|          4.4|         2.9|          1.4|         0.2|Iris-setosa|
| 10|          4.9|         3.1|          1.5|         0.1|Iris-se

### Step 1.3: Data Preprocessing (Optional for AutoML)

Databricks AutoML handles most preprocessing automatically, including:
- Missing value imputation
- Feature engineering
- Train/validation/test splitting

We just need to verify the target column is correctly identified and exclude any ID columns if present.


In [3]:
# Check for ID column to exclude
id_cols = ["Id", "id", "ID"] if any(col in df.columns for col in ["Id", "id", "ID"]) else None
exclude_cols = [col for col in id_cols if col in df.columns] if id_cols else None

print(f"Columns to exclude from features: {exclude_cols}")
print(f"Target column: Species")
print(f"Feature columns: {[c for c in df.columns if c != 'Species' and (exclude_cols is None or c not in exclude_cols)]}")

# Check for missing values (AutoML will handle imputation automatically)
print("\nMissing values per column:")
for col_name in df.columns:
    missing_count = df.filter(F.col(col_name).isNull()).count()
    if missing_count > 0:
        print(f"  {col_name}: {missing_count}")
    else:
        print(f"  {col_name}: 0 (no missing values)")


Columns to exclude from features: ['Id']
Target column: Species
Feature columns: ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']

Missing values per column:
  Id: 0 (no missing values)
  SepalLengthCm: 0 (no missing values)
  SepalWidthCm: 0 (no missing values)
  PetalLengthCm: 0 (no missing values)
  PetalWidthCm: 0 (no missing values)
  Species: 0 (no missing values)


## Phase 2: MLflow Configuration


### Step 2.1: Initialize MLflow

MLflow autologging will automatically track:
- Parameters
- Metrics
- Model artifacts
- Model signatures


In [4]:
db_connect_env = "field-eng"

# Set MLflow tracking URI to Databricks
mlflow.set_tracking_uri(f"databricks://{db_connect_env}")

# Set experiment name
# experiment_name = "iris_demo"
# mlflow.set_experiment(experiment_name)

# Enable autologging for scikit-learn models
mlflow.sklearn.autolog()

# Verify connection
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
# print(f"MLflow experiment: {mlflow.get_experiment_by_name(experiment_name)}")
print("MLflow autologging enabled for scikit-learn")

# if artifact download sometimes “hangs”, make it fail fast instead of spinning:
os.environ["MLFLOW_HTTP_REQUEST_TIMEOUT"] = "45"
os.environ["DATABRICKS_CONFIG_PROFILE"] = db_connect_env  # ensures artifact repo uses the same profile

MLflow tracking URI: databricks://field-eng
MLflow autologging enabled for scikit-learn


## Phase 3: Databricks AutoML Model Selection

Databricks AutoML will automatically:
- Test multiple algorithms (sklearn, LightGBM, potentially XGBoost)
- Perform hyperparameter tuning
- Split data into train/validation/test sets
- Log all trials to MLflow
- Generate trial notebooks for the best model


### Step 3.2: Run Databricks AutoML Classification (Not Available via DBConnect)


In [ ]:
# Run Databricks AutoML classification
# AutoML will test multiple algorithms and log all runs to MLflow
summary = databricks.automl.classify(
    dataset=df,  # Spark DataFrame
    target_col="Species",
    primary_metric="f1",  # Options: "f1", "log_loss", "precision", "accuracy", "roc_auc"
    experiment_name="iris_demo",  # Align with Phase 2 experiment
    exclude_cols=exclude_cols,  # Exclude ID column if present
    timeout_minutes=30,  # Adjust based on time constraints (default is 120)
)

print("AutoML run completed!")
print(f"Number of trials: {len(summary.trials)}")


## Step 4: Load The Best Model & Run Predictions


### Step 4.1 Load best_model

In [8]:
client = MlflowClient()
runs = client.search_runs("4362073218491067", order_by=["metrics.val_f1_score DESC"], max_results=1)
best_run = runs[0] if runs else None
best_model_type = best_run.data.tags["model_type"]
best_model_name = best_run.data.tags["mlflow.runName"]
model_uri = f"models:/{best_run.outputs.model_outputs[0].model_id}"
best_model = mlflow.pyfunc.load_model(model_uri)


### Step 4.2 Make a Prediction

In [16]:
mlflow.models.predict(
    model_uri=model_uri,
    input_data=best_model.input_example,
    env_manager="uv",
)

2025/11/11 14:22:49 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


2025/11/11 14:22:52 INFO mlflow.utils.virtualenv: Creating a new environment in /tmp/virtualenv_envs/mlflow-a5dbc391bb593c3c716e6af7aee172342701b95d with python version 3.12.3 using uv
Using CPython 3.12.3 interpreter at: /opt/homebrew/anaconda3/envs/py312dbx/bin/python3.12
Creating virtual environment at: /tmp/virtualenv_envs/mlflow-a5dbc391bb593c3c716e6af7aee172342701b95d
Activate with: source /tmp/virtualenv_envs/mlflow-a5dbc391bb593c3c716e6af7aee172342701b95d/bin/activate
2025/11/11 14:22:53 INFO mlflow.utils.virtualenv: Installing dependencies
Using Python 3.12.3 environment at: /tmp/virtualenv_envs/mlflow-a5dbc391bb593c3c716e6af7aee172342701b95d
Resolved 3 packages in 603ms
Prepared 3 packages in 658ms
Installed 3 packages in 12ms
 + pip==25.0.1
 + setuptools==74.0.0
 + wheel==0.45.1
Using Python 3.12.3 environment at: /tmp/virtualenv_envs/mlflow-a5dbc391bb593c3c716e6af7aee172342701b95d
Resolved 78 packages in 1.99s
   Building lz4==4.3.2
   Building psutil==5.9.0
      Built psu

{"predictions": ["Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa"]}